# 04 - Petit U-Net : radar Sentinel-1 et entraînement plus long

Le petit U-Net est le modèle retenu (léger, peu de fausses alertes). Deux questions :

1. **Entraîner plus longtemps aide-t-il ?** Dans le notebook 03, l'IoU de validation plafonnait
   dès l'époque 10 environ, alors que la perte d'entraînement baissait encore.
2. **Le radar aide-t-il face aux nuages ?** Sentinel-1 est un radar : il voit à travers les nuages,
   et l'eau calme y apparaît très sombre. On ajoute ses 2 bandes (VV, VH) aux 13 bandes de Sentinel-2.

Trois entraînements, **avec la même graine aléatoire**, pour que seule la variable étudiée change :

| Expérience | Bandes | Époques |
|---|---|---|
| A (référence) | Sentinel-2 (13) | 30 |
| B | Sentinel-2 (13) | 60 |
| C | Sentinel-2 + Sentinel-1 (15) | 30 |

Exécution : Colab avec GPU T4, *Exécution > Tout exécuter* (environ 25 minutes).
Garder l'ordinateur éveillé pendant l'exécution.

In [ ]:
import os, subprocess, sys

if not os.path.isdir("one-look"):
    subprocess.run(["git", "clone", "-q", "https://github.com/lemembre05/one-look.git"], check=True)
else:
    subprocess.run(["git", "-C", "one-look", "pull", "-q"], check=True)
sys.path.insert(0, "one-look")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rasterio"], check=True)

import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
from src import donnees, modeles, entrainement, alerte
from src.metriques import scores_alerte

noms = donnees.telecharger("data", radar=True)
APPAREIL = "cuda" if torch.cuda.is_available() else "cpu"
print(len(noms), "images | calcul sur", APPAREIL, torch.cuda.get_device_name(0) if APPAREIL == "cuda" else "")
if APPAREIL == "cpu":
    print("ATTENTION : pas de GPU. Exécution > Modifier le type d'exécution > GPU T4, puis relancer.")

## 1. Chargement : 13 bandes optiques + 2 bandes radar

Chaque image est stockée une seule fois avec ses 15 bandes (entiers 16 bits ; radar en centièmes de dB).
Les modèles « Sentinel-2 seul » lisent les 13 premières bandes de la même image, sans copie.

In [ ]:
images, labels, perms = [], [], []
for nom in noms:
    s2, label, perm = donnees.lire_image(nom, "data")
    s1 = donnees.lire_radar(nom, "data")
    images.append(np.concatenate([s2, s1 * 100]).astype(np.int16))
    labels.append(label.astype(np.int8))
    perms.append(perm)
images_s2 = [x[:13] for x in images]          # vues sur les 13 bandes optiques (pas de copie)

df = pd.DataFrame({"image": noms, "pays": [donnees.pays(n) for n in noms], "partie": [donnees.partie(n) for n in noms]})
df["part_inondee"] = [donnees.part_inondee_vraie(l.astype(np.float32), p) for l, p in zip(labels, perms)]
df["part_nuages"] = [float((l < 0).mean()) for l in labels]
SEUIL_ALERTE = 0.05
df["alerte_vraie"] = df["part_inondee"] >= SEUIL_ALERTE

idx = {p: df.index[df["partie"] == p].tolist() for p in ["train", "validation", "test"]}
moyenne, ecart = entrainement.statistiques_bandes(images, idx["train"])
print({p: len(i) for p, i in idx.items()})
print("Radar VV / VH, moyenne sur l'entraînement :", (moyenne[13:] / 100).round(1), "dB")

### À quoi ressemble le radar ?

Image optique, bande radar VV (l'eau est sombre) et eau annotée, sur l'image très nuageuse
du Ghana que tous les modèles optiques ratent.

In [ ]:
for nom in ["Ghana_866994", "Ghana_313799"]:
    if nom in noms:
        k = noms.index(nom)
        fig, ax = plt.subplots(1, 3, figsize=(12, 4))
        ax[0].imshow(donnees.rgb(images_s2[k].astype(np.float32))); ax[0].set_title(f"Optique : {nom}")
        ax[1].imshow(images[k][13] / 100, cmap="gray", vmin=-25, vmax=0); ax[1].set_title("Radar VV (dB)")
        ax[2].imshow((labels[k] == 1) & ~perms[k], cmap="Blues"); ax[2].set_title(f"Annoté : {df['part_inondee'][k]:.1%}")
        for a in ax: a.axis("off")
        plt.show()

## 2. Les trois entraînements

Même recette que dans le notebook 03 (découpes 256 x 256, augmentation, entropie croisée + Dice,
meilleure époque sur la validation). Seuls changent le nombre de bandes et le nombre d'époques.

In [ ]:
EPOQUES_COURT, EPOQUES_LONG = (30, 60) if APPAREIL == "cuda" else (2, 3)

experiences = {
    "A : S2, 30 époques": dict(canaux=13, epoques=EPOQUES_COURT),
    "B : S2, 60 époques": dict(canaux=13, epoques=EPOQUES_LONG),
    "C : S2 + S1, 30 époques": dict(canaux=15, epoques=EPOQUES_COURT),
}
reseaux, historiques, donnees_de = {}, {}, {}
os.makedirs("modeles", exist_ok=True)
for nom, cfg in experiences.items():
    c = cfg["canaux"]
    imgs = images if c == 15 else images_s2
    m = modeles.Normalise(modeles.PetitUNet(canaux=c), moyenne[:c], ecart[:c])
    n_param = f"{modeles.nb_parametres(m):,}".replace(",", " ")
    print(f"\n=== {nom} ({c} bandes, {n_param} paramètres) ===")
    historiques[nom] = entrainement.entrainer(m, imgs, labels, idx["train"], idx["validation"],
                                              APPAREIL, epoques=cfg["epoques"])
    reseaux[nom], donnees_de[nom] = m, imgs
    torch.save(m.state_dict(), f"modeles/petit_unet_{'radar' if c == 15 else 's2'}_{cfg['epoques']}ep.pt")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for nom, h in historiques.items():
    ax[0].plot(h["epoque"], h["perte"], label=nom)
    ax[1].plot(h["epoque"], h["iou_validation"], label=nom)
ax[0].set_title("Perte d'entraînement"); ax[1].set_title("IoU de l'eau, validation")
for a in ax: a.set_xlabel("époque"); a.legend()
plt.show()
for nom, h in historiques.items():
    print(f"{nom} : meilleure IoU de validation {h['iou_validation'].max():.3f} "
          f"(époque {int(h.loc[h['iou_validation'].idxmax(), 'epoque'])})")

## 3. Alertes et cartes d'eau

Même protocole que pour la baseline : seuil `t_alerte` choisi sur l'entraînement, scores sur la
validation et le test.

In [ ]:
train = (df["partie"] == "train").to_numpy()
tous = df.index.tolist()
s2f = {k: images_s2[k].astype(np.float32) for k in tous}

parts = {"Baseline NDWI": np.array([alerte.part_eau_ndwi(s2f[k], perms[k]) for k in tous])}
seuils = {"Baseline NDWI": alerte.BASELINE_T_ALERTE}
probs = {}
for nom, m in reseaux.items():
    probs[nom] = entrainement.predire(m, donnees_de[nom], tous, APPAREIL)
    parts[nom] = np.array([alerte.part_eau_reseau(probs[nom][k], s2f[k], perms[k]) for k in tous])
    seuils[nom], _ = alerte.choisir_seuil_alerte(df["alerte_vraie"][train], parts[nom][train], SEUIL_ALERTE)

def ligne(nom, sel):
    pred = parts[nom] >= seuils[nom]
    return scores_alerte(df["alerte_vraie"][sel], pred[sel])

lignes = []
for nom in parts:
    for p in ["validation", "test"]:
        sel = (df["partie"] == p).to_numpy()
        iou = entrainement.iou_eau(probs[nom], labels, idx[p]) if nom in probs else None
        lignes.append({"modele": nom, "partie": p, "t_alerte": seuils[nom], "iou_eau": iou, **ligne(nom, sel)})
scores = pd.DataFrame(lignes)
print(scores[["modele", "partie", "t_alerte", "iou_eau", "precision", "rappel", "f1",
              "taux_fausse_alerte", "fausses_alertes", "manquees"]].round(3).to_string(index=False))

## 4. Et sous les nuages ?

Images de validation et de test dont au moins 20 % de la surface est sans annotation
(le plus souvent à cause des nuages).

In [ ]:
nuageux = ((df["partie"] != "train") & (df["part_nuages"] >= 0.2)).to_numpy()
print(f"{nuageux.sum()} images nuageuses en validation + test, dont {int(df['alerte_vraie'][nuageux].sum())} inondées\n")
lignes = []
for nom in parts:
    s = ligne(nom, nuageux)
    lignes.append({"modele": nom, "inondees_detectees": f"{s['vrais_positifs']} / {s['inondees']}",
                   "fausses_alertes": s["fausses_alertes"], "f1": s["f1"]})
print(pd.DataFrame(lignes).to_string(index=False))

def montrer(k):
    s2 = s2f[k]
    vrai = (labels[k] == 1) & ~perms[k]
    fig, ax = plt.subplots(1, 4, figsize=(16, 4))
    ax[0].imshow(donnees.rgb(s2)); ax[0].set_title(noms[k])
    for a, nom in zip(ax[1:3], ["A : S2, 30 époques", "C : S2 + S1, 30 époques"]):
        a.imshow((probs[nom][k] > 0.5) & donnees.pixels_utiles(s2, perms[k]), cmap="Blues")
        a.set_title(f"{nom[:1]} : {parts[nom][k]:.1%}")
    ax[3].imshow(vrai, cmap="Blues"); ax[3].set_title(f"Annoté : {df['part_inondee'][k]:.1%}")
    for a in ax: a.axis("off")
    plt.show()

for nom in ["Ghana_866994", "Ghana_313799", "Pakistan_246718", "Pakistan_211386"]:
    if nom in noms:
        montrer(noms.index(nom))

## 5. Tableau d'ablation (taille, vitesse, qualité)

In [ ]:
lignes = []
for nom, m in reseaux.items():
    c = experiences[nom]["canaux"]
    v = scores[(scores["modele"] == nom) & (scores["partie"] == "validation")].iloc[0]
    t = scores[(scores["modele"] == nom) & (scores["partie"] == "test")].iloc[0]
    lignes.append({"experience": nom, "parametres": modeles.nb_parametres(m),
                   "taille_Mo": round(modeles.taille_mo(m), 2),
                   "ms_par_image_1_coeur": round(entrainement.latence_cpu_ms(m, canaux=c), 0),
                   "iou_val": round(v["iou_eau"], 3), "f1_val": v["f1"], "fausses_alertes_val": v["taux_fausse_alerte"],
                   "iou_test": round(t["iou_eau"], 3), "f1_test": t["f1"], "fausses_alertes_test": t["taux_fausse_alerte"]})
ablation = pd.DataFrame(lignes).set_index("experience")
print(ablation.to_string())
os.makedirs("resultats", exist_ok=True)
ablation.to_csv("resultats/ablation_04.csv")

## À retenir

- Comparer A et B dit si le temps d'entraînement était la limite.
- Comparer A et C dit ce qu'apporte le radar, en particulier sur les images nuageuses.
- Avec 18 alertes en validation et 24 en test, un écart d'une ou deux images n'est pas significatif :
  l'IoU par pixel (des millions de pixels) est la mesure la plus stable pour comparer les cartes d'eau.